In [18]:
#This code will wrangle the data obtained from both the TCGA directly and also from related studies applied to this project
#we will need to organize all the tcga IDs from each file to make sure that we have X,Y,Z for our CI model with matching IDs for accurate prediction
#Initially we did not have the Y or the RNA seq data for the tcga IDs, we will have to gather what IDs we have first from the X data file that has all the cases
#we  will use cross matching the IDs with the data files given then based on that 

#this section is for installing all the necessary modules within the bluebear jupyter environment
#you will need to restart the kernal after install
#!pip install --user pandas
#!pip install --user openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 14.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [13]:


#Z Channel Data Cross Referencing sample IDs for X and s2 aneuploidy/purity scores

#First from observing the all_cases file, the case IDs match some of the sample column rows in the Taylor et. al 2018 table 2 data. 
#Next we can also observe from the all_cases file the case IDs match some of the array columns 
#Lastly for the all_cases file case IDs we see some matching in the PANCAN armcallsand aneuploidy in the sample column (TCGA-18-3407)
#from these observations we will use the case column in the all_cases file then cross reference these IDs to the table s2 aneuploidy/purity scores

#lets import all the necessary modules/add ons
import pandas as pd

#-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#first lets make a list of all unique rows in the case column within the all_cases file

#define the path of the all_cases file
path_all_cases = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/all_cases_ascat2_TCGA.csv'

#load the data into a pandas DataFrame
df_cases = pd.read_csv(path_all_cases)

#check if column names match exact column containing the TCGA IDs
print("Columns in all_cases:", df_cases.columns.tolist())
print(df_cases.head(2))

#then extract unique case IDs from the X/copy-number dataset
unique_cases_x = df_cases['Case'].unique()
print(f"Total unique cases found in X data: {len(unique_cases_x)}")
#print(unique_cases_x)

#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#now that we have the unique cases_x
#we would need to match the case ID to the samples column in the table s2 aneuploidy/purity samples

#define the path of the s2 table excel and output path
path_taylor_s2 = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/Z/tcga_taylor2018_table_s2_sample_aneuploidy_purity.xlsx'
taylor_s2_csv_output_path = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/Z/tcga_taylor2018_table_s2_sample_aneuploidy_purity.csv'

#first lets convert the xlsx file to a csv file
taylor_s2_df = pd.read_excel(path_taylor_s2)

#now save it as a csv file no header (ignore top row) and no index
taylor_s2_df.to_csv(taylor_s2_csv_output_path,header=None, index=False)

#now read the csv file in for the table s2
taylor_s2_df = pd.read_csv(taylor_s2_csv_output_path)

#check headers and columns of new csv file
#print("Columns in table s2:", taylor_s2_df.columns.tolist())

#now lets extract the unique sample IDs
#from TCGA barcode naming extensions the last extension refers to the sample type
#we would want solid tumors or -01 for our pipeline

#we can filter by these sample types by adding a column next to the samples to
#identify the solid tumor samples, from there we can perform sample matching with the all_cases

#create column inf dataframe with only -01 sample type
#we will use apply with lambda x applied to every row of Sample column to slice grabbing the 4th element from the split list -01
#strictly with the first two characters -01 not -01A,B,C etc
#only have the the legnth of the split of sample IDs greater than equal to 4

taylor_s2_df['sample_type_code'] = taylor_s2_df['Sample'].apply(lambda x: x.split('-')[3][:2] if len(x.split('-'))>=4 else None)

#now filter for primary solid tumors using the code 01
s2_primary_tumors_df = taylor_s2_df[taylor_s2_df['sample_type_code'] == '01'].copy()

# print(f"Total rows before filtering: {len(s2_primary_tumors_df)}")
# print(f"Primary solid tumor samples ('01'): {len(s2_primary_tumors_df)}")
# print(s2_primary_tumors_df.head())

#next let's grab all the sample IDs and match them with the all_cases IDs unique_cases_x
# add another column called ID for filtering out the -01 at the end called ID
s2_primary_tumors_df['ID'] = s2_primary_tumors_df['Sample'].str[:-3]

#get unique case IDs from unique cases X dataset
common_patient_IDs_s2 = set(s2_primary_tumors_df['ID']).intersection(set(unique_cases_x))

#print(common_patient_IDs_s2)
print(f"The number of common patient IDs in s2 is: {len(common_patient_IDs_s2)}")

#now create filtered dfs for s2 and X given common IDs
df_x_filtered = df_cases[df_cases['Case'].isin(common_patient_IDs_s2)].copy()
df_z_filtered = s2_primary_tumors_df[s2_primary_tumors_df['ID'].isin(common_patient_IDs_s2)].copy()

print(df_x_filtered.head(2))
print(df_z_filtered.head(2))












Columns in all_cases: ['GDC_Aliquot', 'Chromosome', 'Start', 'End', 'Copy_Number', 'Major_Copy_Number', 'Minor_Copy_Number', 'Case', 'Tumor_Type', 'Sample_ID', 'Sample_UUID', 'Tissue_Type']
                            GDC_Aliquot Chromosome     Start         End  \
0  46ad10f0-8259-4984-a331-2c004d0e20e4       chr1     62920  62750154.0   
1  46ad10f0-8259-4984-a331-2c004d0e20e4       chr1  62750700  63026943.0   

   Copy_Number  Major_Copy_Number  Minor_Copy_Number          Case Tumor_Type  \
0            2                  1                  1  TCGA-EX-A1H6  TCGA-CESC   
1            2                  2                  0  TCGA-EX-A1H6  TCGA-CESC   

          Sample_ID                           Sample_UUID    Tissue_Type  
0  TCGA-EX-A1H6-01B  fe018d31-eee8-4335-96a1-95eceabfadf1  Primary Tumor  
1  TCGA-EX-A1H6-01B  fe018d31-eee8-4335-96a1-95eceabfadf1  Primary Tumor  
Total unique cases found in X data: 10763
The number of common patient IDs in s2 is: 9779
                      

In [ ]:
#Z Channel Data Cross Referencing sample IDs for X and PANCAN ArmCallsAndAneuploidy
#Think chunk of code will 

